<a href="https://colab.research.google.com/github/YASHASWINIKSHRESTHA/AI-SUMMARIZER-EXTENSION/blob/main/HYPERSPECTRAL_BTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [2]:
!pip install numpy matplotlib scipy

In [3]:
!mkdir -p HSI_Project/models
!mkdir -p HSI_Project/utils
!mkdir -p HSI_Project/results

In [4]:
%%writefile HSI_Project/utils/dataset.py
import torch
from torch.utils.data import Dataset
import numpy as np

def normalize_hsi(hsi):
    hsi = hsi.astype(np.float32)
    return (hsi - hsi.min()) / (hsi.max() - hsi.min() + 1e-8)

def extract_patches(hsi, patch_size=32, stride=8):
    patches = []
    H, W, B = hsi.shape

    for i in range(0, H - patch_size + 1, stride):
        for j in range(0, W - patch_size + 1, stride):
            patches.append(hsi[i:i+patch_size, j:j+patch_size, :])

    patches = np.array(patches)
    print("Total patches:", patches.shape[0])
    return patches

class HSIStegoDataset(Dataset):
    def __init__(self, patches, payload_rate=0.3):
        self.patches = patches
        self.payload_rate = payload_rate

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        cover = self.patches[idx]
        cover = torch.tensor(cover, dtype=torch.float32).permute(2,0,1)

        num_bits = int(self.payload_rate * cover.shape[1] * cover.shape[2])
        payload = torch.zeros(1, cover.shape[1], cover.shape[2])
        payload.view(-1)[:num_bits] = torch.randint(0,2,(num_bits,),dtype=torch.float32)

        return cover, payload

Writing HSI_Project/utils/dataset.py


In [5]:
%%writefile HSI_Project/models/encoder_decoder.py
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, bands):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(bands + 1, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, bands, 3, padding=1)
        )

        self.alpha = 0.01

    def forward(self, cover, payload):
        x = torch.cat([cover, payload], dim=1)
        residual = torch.tanh(self.net(x))
        return cover + self.alpha * residual


class Decoder(nn.Module):
    def __init__(self, bands):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(bands, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

Writing HSI_Project/models/encoder_decoder.py


In [6]:
%%writefile HSI_Project/utils/metrics.py
import torch

def psnr(x, y):
    mse = torch.mean((x - y) ** 2)
    return 10 * torch.log10(1 / (mse + 1e-8))

def sam(x, y):
    x = x.permute(0,2,3,1).reshape(-1, x.shape[1])
    y = y.permute(0,2,3,1).reshape(-1, y.shape[1])

    dot = torch.sum(x*y, dim=1)
    norm = torch.norm(x, dim=1) * torch.norm(y, dim=1)
    cos = dot / (norm + 1e-8)
    cos = torch.clamp(cos, -1, 1)

    return torch.mean(torch.acos(cos))

Writing HSI_Project/utils/metrics.py


In [7]:
%%writefile HSI_Project/main.py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from utils.dataset import normalize_hsi, extract_patches, HSIStegoDataset
from models.encoder_decoder import Encoder, Decoder
from utils.metrics import psnr, sam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load Dataset (Upload indianpinearray.npy manually to Colab first)
hsi = np.load("indianpinearray.npy")
hsi = normalize_hsi(hsi)
patches = extract_patches(hsi)

bands = patches.shape[3]

payload_rates = [0.1, 0.3, 0.5, 0.8]
results = []

for rate in payload_rates:

    dataset = HSIStegoDataset(patches, payload_rate=rate)
    loader = DataLoader(dataset, batch_size=8, shuffle=True)

    encoder = Encoder(bands).to(device)
    decoder = Decoder(bands).to(device)

    optimizer = optim.Adam(
        list(encoder.parameters()) + list(decoder.parameters()),
        lr=1e-4
    )

    mse_loss = nn.MSELoss()
    bce_loss = nn.BCELoss()

    # Train
    for epoch in range(15):
        for cover, payload in loader:

            cover = cover.to(device)
            payload = payload.to(device)

            stego = encoder(cover, payload)

            # Gaussian Noise Channel
            noise = torch.randn_like(stego) * 0.005
            stego_noisy = stego + noise

            recovered = decoder(stego_noisy)

            loss = 2 * mse_loss(stego, cover) + 5 * bce_loss(recovered, payload)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate
    encoder.eval()
    decoder.eval()

    total_psnr = 0
    total_sam = 0
    total_ber = 0
    total_energy = 0

    with torch.no_grad():
        for cover, payload in loader:

            cover = cover.to(device)
            payload = payload.to(device)

            stego = encoder(cover, payload)
            recovered = decoder(stego)

            total_psnr += psnr(stego, cover).item()
            total_sam += sam(stego, cover).item()
            total_energy += torch.mean((stego-cover)**2).item()

            pred = (recovered > 0.5).float()
            total_ber += torch.mean(torch.abs(pred - payload)).item()

    avg_psnr = total_psnr / len(loader)
    avg_sam = total_sam / len(loader)
    avg_ber = total_ber / len(loader)
    avg_energy = total_energy / len(loader)

    print(f"\nPayload {rate}")
    print("PSNR:", avg_psnr)
    print("SAM:", avg_sam)
    print("BER:", avg_ber)
    print("Energy:", avg_energy)

    results.append((rate, avg_psnr, avg_sam, avg_ber, avg_energy))

# Plot
rates = [r[0] for r in results]
psnr_vals = [r[1] for r in results]
sam_vals = [r[2] for r in results]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(rates, psnr_vals, marker='o')
plt.title("PSNR vs Payload")

plt.subplot(1,2,2)
plt.plot(rates, sam_vals, marker='o')
plt.title("SAM vs Payload")

plt.show()

Writing HSI_Project/main.py


In [9]:
!python HSI_Project/main.py

Using device: cuda
Total patches: 225

Payload Rate: 0.1
PSNR: 40.021
SAM: 0.02534
BER: 0.0495
Energy: 9.951210692330885e-05

Payload Rate: 0.3
PSNR: 40.489
SAM: 0.02915
BER: 0.12863
Energy: 8.93479045771528e-05

Payload Rate: 0.5
PSNR: 41.142
SAM: 0.03084
BER: 0.17634
Energy: 7.687128163524903e-05

Payload Rate: 0.8
PSNR: 42.515
SAM: 0.02428
BER: 0.24716
Energy: 5.603753925242927e-05
Figure(1200x800)
